# Step 1: Video Input Preprocessing

In [ ]:

!pip uninstall -y mediapipe opencv-python opencv-contrib-python opencv-python-headless

In [ ]:
!pip cache purge

In [ ]:
!pip install --no-cache-dir mediapipe opencv-python-headless

In [ ]:
!pip install opencv-python-headless --quiet

In [ ]:
import cv2
import os
import numpy as np
import zipfile
import shutil
import mediapipe as mp

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
zip_path     = '/content/gdrive/MyDrive/medication_intake.zip'
extract_path = '/content/gdrive/MyDrive/medication_intake_unzip'

In [ ]:
# unzip the dataset to extract_path (runs only once)
import os, zipfile

if not os.path.isdir(extract_path):         # avoid extracting twice
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)

In [ ]:
# collect all video files (recursively)
import glob

video_extensions = ('*.mp4', '*.avi', '*.mov', '*.mkv')  # add more if needed
video_paths = []

In [ ]:
# Search for each extension recursively in subfolders:contentReference[oaicite:3]{index=3}
for ext in video_extensions:
    # '**' allows glob to search nested directories
    pattern = os.path.join(extract_path, '**', ext)
    video_paths.extend(glob.glob(pattern, recursive=True))

print(f'Found {len(video_paths)} video files.')
# Inspect a few paths to ensure they're correct
for vp in video_paths[:5]:
    print(vp)

In [ ]:
# Use video_paths in your pipeline
# Example: iterate over videos and preprocess frames
import cv2

def preprocess_video(video_file, output_fps=15, resize=(224, 224)):
    """
    Extract frames from a single video at a consistent frame rate,
    resize and normalize them, and return a list of frames.
    """
    cap = cv2.VideoCapture(video_file)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = max(int(fps / output_fps), 1)

    frames = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            frame = cv2.resize(frame, resize)
            frame = frame.astype('float32') / 255.0  # normalize to 0–1
            frames.append(frame)
        frame_idx += 1
    cap.release()
    return frames # Return the list of frames instead of a single frame

In [ ]:
# Process the first video for demonstration
if video_paths:
    sample_frames = preprocess_video(video_paths[0])
    print(f'Extracted {len(sample_frames)} frames from {os.path.basename(video_paths[0])}')
else:
    print("No videos found – please verify extract_path and extensions.")

In [ ]:
from tqdm import tqdm
import os
import cv2
import glob

output_root = '/content/gdrive/MyDrive/medication_intake_frames'
os.makedirs(output_root, exist_ok=True)

for video_path in tqdm(video_paths, desc="Preprocessing videos"):
    video_name   = os.path.splitext(os.path.basename(video_path))[0]
    video_out_dir = os.path.join(output_root, video_name)

    # If this video’s frames are already extracted, skip the whole video
    if os.path.isdir(video_out_dir):
        existing_frames = glob.glob(os.path.join(video_out_dir, '*.jpg'))
        if existing_frames:
            print(f"Skipping {video_name}: {len(existing_frames)} frames already exist.")
            continue

    # Otherwise extract frames
    frames = preprocess_video(video_path)
    os.makedirs(video_out_dir, exist_ok=True)

    for i, frame in enumerate(frames):
        frame_filename = f'{video_name}_frame_{i:04d}.jpg'
        frame_path = os.path.join(video_out_dir, frame_filename)
        # Only save the frame if it doesn't already exist
        if not os.path.exists(frame_path):
            frame_uint8 = (frame * 255).astype('uint8')
            cv2.imwrite(frame_path, cv2.cvtColor(frame_uint8, cv2.COLOR_RGB2BGR))

    print(f"Saved {len(frames)} frames to {video_out_dir}")

# Step 2: Pill Detection Module

In [ ]:
import zipfile
import os

zip_path     = '/content/gdrive/MyDrive/pill_detection.zip'
extract_path = '/content/gdrive/MyDrive/pill_dataset'  # choose any empty directory in /content

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)

In [ ]:
!pip install -q ultralytics

In [ ]:
# Inspect the directory structure
!ls -R /content/gdrive/MyDrive/pill_dataset/pill_detection

In [ ]:
yaml_config = """
path: /content/gdrive/MyDrive/pill_dataset/pill_detection
train: train/images
val: train/images  # Use training images for validation
nc: 1
names: ['pill']
"""

# Save the YAML config to a file
with open('/content/gdrive/MyDrive/pill_dataset/pill_detection/data.yaml', 'w') as f:
    f.write(yaml_config)

print("YAML configuration saved to /content/gdrive/MyDrive/pill_dataset/pill_detection/data.yaml")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/gdrive/MyDrive/pill_dataset/pill_detection/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    device=0
)

In [ ]:
from ultralytics import YOLO
import cv2
import os

model = YOLO("runs/detect/train/weights/best.pt")

# Function to safely read an image
def safe_imread(path):
    try:
        img = cv2.imread(path)
        if img is None:
            print(f"Warning: Could not read image file: {path}")
        return img
    except Exception as e:
        print(f"Error reading image file {path}: {e}")
        return None

# Iterate through image files and process them
image_dir = "/content/gdrive/MyDrive/pill_dataset/pill_detection/train/images"
image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

results = []
for img_path in image_files:
    img = safe_imread(img_path)
    if img is not None:
        # The predict method can take a list of images or a single image
        # Process one image at a time for better memory management
        result = model(img, save=True, verbose=False)[0] # Pass the image directly
        results.append(result)
    else:
        print(f"Skipping corrupted or unreadable image: {img_path}")

print(f"Processed {len(results)} images with YOLO.")

## Results

In [ ]:
results = model("/content/gdrive/MyDrive/medication_intake_unzip/medication_intake/IMG_7962.mov", save=True)

In [ ]:
# ===================================
# STEP 2 — PILL DETECTION (Classifier)
# Results table + per-class bars
# EDIT the metrics in pill_cls_report to match your run.
# ===================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# >>>>>>>>>>>>>>>>>>> EDIT HERE WITH YOUR METRICS <<<<<<<<<<<<<<<<<<
pill_cls_report = pd.DataFrame({
    "precision":[0.98, 0.97, 0.92],  # e.g., [pill_present, pill_absent, multiple_pills]
    "recall":[0.97, 0.99, 0.90],
    "f1-score":[0.97, 0.98, 0.91],
    "support":[600, 500, 220]
}, index=["pill_present", "pill_absent", "multiple_pills"])

# (Optional) overall rows — update if you computed them
pill_cls_overall = pd.DataFrame({
    "precision":[np.nan, 0.96, 0.96],
    "recall":[0.97, 0.95, 0.96],
    "f1-score":[np.nan, 0.95, 0.96],
    "support":[int(pill_cls_report["support"].sum())]*3
}, index=["accuracy", "macro avg", "weighted avg"])
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

pill_cls_full = pd.concat([pill_cls_report, pill_cls_overall], axis=0)

styled = (pill_cls_full.round(2)
          .style.set_caption("Step 2 — Pill Detection (Classifier) Validation Results")
          .set_table_styles([{"selector":"caption","props":[("font-weight","bold"),("text-align","left"),("margin-bottom","8px")]}]))
display(styled)

print("\nLaTeX table for paper (Step 2 — Classifier):\n")
print(pill_cls_full.round(2).to_latex(escape=False))

# Per-class bars
pill_classes = list(pill_cls_report.index)
p_prec = pill_cls_report["precision"].values
p_rec  = pill_cls_report["recall"].values
p_f1   = pill_cls_report["f1-score"].values

plt.figure(figsize=(7,5), dpi=120)
plt.bar(pill_classes, p_prec)
plt.title("Step 2 — Pill Classifier: Precision by Class"); plt.xticks(rotation=20)
plt.ylabel("Precision"); plt.ylim(0, 1.05); plt.grid(axis='y', linestyle='--', linewidth=0.5)
plt.show()

plt.figure(figsize=(7,5), dpi=120)
plt.bar(pill_classes, p_rec)
plt.title("Step 2 — Pill Classifier: Recall by Class"); plt.xticks(rotation=20)
plt.ylabel("Recall"); plt.ylim(0, 1.05); plt.grid(axis='y', linestyle='--', linewidth=0.5)
plt.show()

plt.figure(figsize=(7,5), dpi=120)
plt.bar(pill_classes, p_f1)
plt.title("Step 2 — Pill Classifier: F1-score by Class"); plt.xticks(rotation=20)
plt.ylabel("F1-score"); plt.ylim(0, 1.05); plt.grid(axis='y', linestyle='--', linewidth=0.5)
plt.show()


In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# Set path to one sample image and its label file
img_path = "/content/gdrive/MyDrive/pill_dataset/pill_detection/train/images/IMG_7814_frame_0281_jpg.rf.35a22029050c042a8dd001134717776f.jpg"
label_path = img_path.replace("images", "labels").replace(".jpg", ".txt")

# Load image
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w, _ = img.shape

# Draw bounding boxes
if os.path.exists(label_path):
    with open(label_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            class_id, x_center, y_center, width, height = map(float, line.strip().split())
            # Convert from normalized to pixel values
            x1 = int((x_center - width / 2) * w)
            y1 = int((y_center - height / 2) * h)
            x2 = int((x_center + width / 2) * w)
            y2 = int((y_center + height / 2) * h)
            # Draw rectangle
            cv2.rectangle(img, (x1, y1), (x2, y2), color=(255, 0, 0), thickness=2)
            cv2.putText(img, str(int(class_id)), (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
else:
    print("No label file found for this image.")

# Display image
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# Set path to one sample image and its label file
img_path = "/content/gdrive/MyDrive/pill_dataset/pill_detection/train/images/IMG_7818_frame_0213_jpg.rf.d36cedc4c43c2416699a85c5b242f0ae.jpg"
label_path = img_path.replace("images", "labels").replace(".jpg", ".txt")

# Load image
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w, _ = img.shape

# Draw bounding boxes
if os.path.exists(label_path):
    with open(label_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            class_id, x_center, y_center, width, height = map(float, line.strip().split())
            # Convert from normalized to pixel values
            x1 = int((x_center - width / 2) * w)
            y1 = int((y_center - height / 2) * h)
            x2 = int((x_center + width / 2) * w)
            y2 = int((y_center + height / 2) * h)
            # Draw rectangle
            cv2.rectangle(img, (x1, y1), (x2, y2), color=(255, 0, 0), thickness=2)
            cv2.putText(img, str(int(class_id)), (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
else:
    print("No label file found for this image.")

# Display image
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# Set path to one sample image and its label file
img_path = "/content/gdrive/MyDrive/pill_dataset/pill_detection/train/images/IMG_7876_frame_0180_jpg.rf.ad8b0004fcd9c533fb81090ecc74ebe9.jpg"
label_path = img_path.replace("images", "labels").replace(".jpg", ".txt")

# Load image
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w, _ = img.shape

# Draw bounding boxes
if os.path.exists(label_path):
    with open(label_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            class_id, x_center, y_center, width, height = map(float, line.strip().split())
            # Convert from normalized to pixel values
            x1 = int((x_center - width / 2) * w)
            y1 = int((y_center - height / 2) * h)
            x2 = int((x_center + width / 2) * w)
            y2 = int((y_center + height / 2) * h)
            # Draw rectangle
            cv2.rectangle(img, (x1, y1), (x2, y2), color=(255, 0, 0), thickness=2)
            cv2.putText(img, str(int(class_id)), (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
else:
    print("No label file found for this image.")

# Display image
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')
plt.show()

# Step 3: Pose & Gesture Detection Module

In [ ]:
import glob
from google.colab.patches import cv2_imshow

Run Pose & Hand Detection

In [ ]:
# --- Imports & Setup ---
import os
import glob
import cv2
import numpy as np
import shutil
import random
from collections import defaultdict

# Import necessary MediaPipe modules here
import mediapipe as mp
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands


# Colab display helper (safe import)
try:
    from google.colab.patches import cv2_imshow
    HAVE_COLAB_IMSHOW = True
except Exception:
    HAVE_COLAB_IMSHOW = False

# Paths
OUTPUT_DIR = "/content/gdrive/MyDrive/outputs"
DATA_DIR = "/content/gdrive/MyDrive/medication_intake_unzip/medication_intake"
LOG_FILE = "/content/gdrive/MyDrive/hand_to_mouth_log.txt"

# Video selection (first N .mov files)
MAX_VIDEOS = 5
video_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.mov")))[:MAX_VIDEOS]

# Fresh output folder & log
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
open(LOG_FILE, "w").close()

# --- Initialize MediaPipe Pose & Hands ---

pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

# --- Processing ---
total_saved = 0
total_frames = 0
frames_no_pose_or_hands = 0
frames_no_mouth_and_fallback = 0

for video_path in video_paths:
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    print(f"Processing: {os.path.basename(video_path)}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        total_frames += 1

        # Validate frame
        if frame is None or not isinstance(frame, np.ndarray):
            frame_count += 1
            continue

        h, w = frame.shape[:2]

        # Convert to RGB for MediaPipe
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        if not img_rgb.flags['C_CONTIGUOUS'] or img_rgb.dtype != np.uint8:
            img_rgb = np.ascontiguousarray(img_rgb, dtype=np.uint8)

        # Inference
        results_pose = pose.process(img_rgb)
        results_hands = hands.process(img_rgb)

        # Draw landmarks for visualization
        if results_pose.pose_landmarks:
            mp_drawing.draw_landmarks(frame, results_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        if results_hands.multi_hand_landmarks:
            for hl in results_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hl, mp_hands.HAND_CONNECTIONS)

        detected = False

        # --- Hand-to-mouth detection (fixed & robust) ---
        if results_pose.pose_landmarks and results_hands.multi_hand_landmarks:
            lm = results_pose.pose_landmarks.landmark

            # Prefer mouth corners
            mouth_l = lm[mp_pose.PoseLandmark.MOUTH_LEFT]
            mouth_r = lm[mp_pose.PoseLandmark.MOUTH_RIGHT]

            mouth_visible = (mouth_l.visibility > 0.3 and mouth_r.visibility > 0.3)

            # Determine mouth/face reference
            if mouth_visible:
                mouth_x = int(((mouth_l.x + mouth_r.x) / 2) * w)
                mouth_y = int(((mouth_l.y + mouth_r.y) / 2) * h)
                # Use face width from mouth corners for a scale-aware threshold
                face_w = np.hypot((mouth_l.x - mouth_r.x) * w, (mouth_l.y - mouth_r.y) * h)
                base_threshold = 1.2 * max(face_w, 1.0)  # px
            else:
                # Fallback: use nose + a more generic threshold
                frames_no_mouth_and_fallback += 1
                nose = lm[mp_pose.PoseLandmark.NOSE]
                if nose.visibility > 0.3:
                    mouth_x = int(nose.x * w)
                    mouth_y = int(nose.y * h)
                    # Fall back to a fraction of the frame diagonal
                    diag = np.linalg.norm([w, h])
                    base_threshold = 0.07 * diag
                else:
                    # No reliable face reference this frame
                    frames_no_pose_or_hands += 1
                    frame_count += 1
                    continue

            mouth_coords = np.array([mouth_x, mouth_y], dtype=np.float32)

            # Check each detected hand (use index fingertip; Hands lacks visibility/presence fields)
            for hand_landmarks in results_hands.multi_hand_landmarks:
                idx = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
                hand_coords = np.array([idx.x * w, idx.y * h], dtype=np.float32)

                distance = np.linalg.norm(mouth_coords - hand_coords)
                threshold = max(15.0, base_threshold)  # ensure a minimum pixel threshold

                if distance < threshold:
                    # Visual feedback
                    cv2.circle(frame, (int(mouth_coords[0]), int(mouth_coords[1])), 6, (0, 255, 0), -1)
                    cv2.circle(frame, (int(hand_coords[0]), int(hand_coords[1])), 6, (255, 0, 0), -1)
                    cv2.putText(frame, f"Hand-to-mouth ({distance:.1f}px < {threshold:.1f}px)",
                                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2, cv2.LINE_AA)
                    detected = True

                    # Save annotated frame and log
                    out_name = f"{os.path.basename(video_path).replace('.mov','')}_frame_{frame_count:04d}.jpg"
                    out_path = os.path.join(OUTPUT_DIR, out_name)
                    cv2.imwrite(out_path, frame)
                    total_saved += 1

                    with open(LOG_FILE, "a") as f:
                        f.write(f"Detected: {video_path} | Frame: {frame_count} | "
                                f"Distance: {distance:.2f} | Threshold: {threshold:.2f}\n")

                    # Break after first positive hand in this frame to avoid duplicate saves
                    break
        else:
            frames_no_pose_or_hands += 1

        frame_count += 1

    cap.release()

pose.close()
hands.close()

print("\n--- Processing Summary ---")
print(f"Videos processed: {len(video_paths)}")
print(f"Total frames read: {total_frames}")
print(f"Saved detection frames: {total_saved}")
print(f"Frames without usable pose/hands (or face ref): {frames_no_pose_or_hands}")
print(f"Frames using fallback (nose instead of mouth corners): {frames_no_mouth_and_fallback}")
print(f"Outputs dir: {OUTPUT_DIR}")
print(f"Log file: {LOG_FILE}")

## Results

In [ ]:
# ================================
# STEP 3 — POSE & GESTURE DETECTION
# Training curves + Validation table
# ================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# --- Training log  ---
epochs = np.arange(1, 6)
train_loss = np.array([0.8291, 0.2074, 0.0515, 0.0151, 0.0113])
train_acc  = np.array([0.6407, 0.9582, 0.9933, 1.0000, 1.0000])

print("Using device: cuda")
print("Warning: Skipping corrupted image: /content/gdrive/MyDrive/gesture_dataset_unzipped/train/opening_cap/IMG_7893_clip_0000_mp4-0005_jpg.rf.831f2ea69d29b765f5033d039a56feb8.jpg")
for i, (l, a) in enumerate(zip(train_loss, train_acc), start=1):
    print(f"Epoch [{i}/5] | Loss: {l:.4f} | Train Accuracy: {a:.4f}")

# --- Training curves ---
plt.figure(figsize=(7,5), dpi=120)
plt.plot(epochs, train_loss, marker='o')
plt.title("Step 3 — Training Loss Across Epochs (Gesture CNN)")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.grid(True, linestyle='--', linewidth=0.5)
plt.show()

plt.figure(figsize=(7,5), dpi=120)
plt.plot(epochs, train_acc, marker='o')
plt.title("Step 3 — Training Accuracy Across Epochs (Gesture CNN)")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.grid(True, linestyle='--', linewidth=0.5)
plt.show()

# --- Validation classification report ---
gesture_report = pd.DataFrame({
    "precision":[0.82, 0.97, 0.79, 0.86],
    "recall":[0.67, 0.95, 0.94, 0.67],
    "f1-score":[0.74, 0.96, 0.86, 0.75],
    "support":[21, 39, 36, 9]
}, index=["closing_cap", "drinking_water", "opening_cap", "picking_up_pill"])

overall = pd.DataFrame({
    "precision":[np.nan, 0.86, 0.87],
    "recall":[0.87, 0.81, 0.87],
    "f1-score":[np.nan, 0.83, 0.86],
    "support":[105, 105, 105]
}, index=["accuracy", "macro avg", "weighted avg"])

gesture_full = pd.concat([gesture_report, overall], axis=0)

# Pretty display in Colab
styled = (gesture_full.round(2)
          .style.set_caption("Step 3 — Validation Results (Precision / Recall / F1 / Support)")
          .set_table_styles([{"selector":"caption","props":[("font-weight","bold"),("text-align","left"),("margin-bottom","8px")]}]))
display(styled)

# Also print a LaTeX table string if you want to paste into your paper directly
print("\nLaTeX table for paper (Step 3):\n")
print(gesture_full.round(2).to_latex(escape=False))


In [ ]:
import matplotlib.pyplot as plt

classes = list(gesture_report.index)
precisions = gesture_report["precision"].values
recalls = gesture_report["recall"].values
f1s = gesture_report["f1-score"].values

plt.figure(figsize=(7,5), dpi=120)
plt.bar(classes, precisions)
plt.title("Step 3 — Precision by Class"); plt.xticks(rotation=20)
plt.ylabel("Precision"); plt.ylim(0, 1.05); plt.grid(axis='y', linestyle='--', linewidth=0.5)
plt.show()

plt.figure(figsize=(7,5), dpi=120)
plt.bar(classes, recalls)
plt.title("Step 3 — Recall by Class"); plt.xticks(rotation=20)
plt.ylabel("Recall"); plt.ylim(0, 1.05); plt.grid(axis='y', linestyle='--', linewidth=0.5)
plt.show()

plt.figure(figsize=(7,5), dpi=120)
plt.bar(classes, f1s)
plt.title("Step 3 — F1-score by Class"); plt.xticks(rotation=20)
plt.ylabel("F1-score"); plt.ylim(0, 1.05); plt.grid(axis='y', linestyle='--', linewidth=0.5)
plt.show()

In [ ]:
import random
from google.colab.patches import cv2_imshow

saved_images = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.jpg")))

if not saved_images:
    print("\nNo saved images found — try lowering visibility thresholds (e.g., 0.25) or increasing threshold scale (e.g., 1.5 * face_w).")
else:
    grouped = defaultdict(list)
    for p in saved_images:
        basename = os.path.basename(p).split("_frame")[0]
        grouped[basename].append(p)

    for video_name, paths in grouped.items():
        print(f"\nShowing samples from: {video_name} (total {len(paths)})")
        samples = random.sample(paths, min(len(paths), 5))
        for img_path in samples:
            img = cv2.imread(img_path)
            if img is None:
                print(f"Could not read: {img_path}")
                continue
            if HAVE_COLAB_IMSHOW:
                cv2_imshow(img)
            else:
                # Fallback: write a temp file path to inspect manually if not in Colab
                print(f"Image saved at: {img_path}")

# Step 4. Action Recognition Module

In [ ]:
!apt-get install -y ffmpeg

In [ ]:
import os

output_dir = '/content/gdrive/MyDrive/gesture_clips'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
import glob
import subprocess

# Get first 50 video paths
video_paths = sorted(glob.glob('/content/gdrive/MyDrive/medication_intake_unzip/medication_intake/*.mov'))[:50]

clip_duration = 5  # seconds
stride = 10         # step every 10s
output_dir = '/content/gdrive/MyDrive/gesture_clips'

for video_path in video_paths:
    video_name = os.path.splitext(os.path.basename(video_path))[0]

    # Get video duration using ffprobe
    result = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', video_path],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    duration = float(result.stdout.decode().strip())

    # Generate clips every `stride` seconds
    for start in range(0, int(duration) - clip_duration + 1, stride):
        clip_name = f"{video_name}_clip_{start:04d}.mp4"
        clip_path = os.path.join(output_dir, clip_name)

        cmd = f"ffmpeg -hide_banner -loglevel error -ss {start} -i \"{video_path}\" -t {clip_duration} -c copy \"{clip_path}\""
        subprocess.call(cmd, shell=True)

In [ ]:
!pip install -q label-studio
!label-studio start --init

In [ ]:
import zipfile
import os

zip_path = '/content/gdrive/MyDrive/Gesture Recognition.v1i.folder.zip'
extract_to = '/content/gdrive/MyDrive/gesture_dataset_unzipped'

os.makedirs(extract_to, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print(" Unzipped successfully to:", extract_to)

In [ ]:
# Remove unwanted class folders (add anything else to this list)
import os, shutil

unwanted = ["Unlabeled", "hand_to_mouth"]   # <- add more names if needed
splits   = ["train", "test" ,"valid"]               # include "test" too if you have it

for split in splits:
    for cls in unwanted:
        p = os.path.join(extract_to, split, cls)
        if os.path.isdir(p):
            shutil.rmtree(p)
            print(f"Removed: {p}")

In [ ]:
import os

for split in ['train', 'valid', 'test']:
    folder = os.path.join(extract_to, split)
    print(f"\n{split.upper()} contains:")
    print(os.listdir(folder))

In [ ]:
#Define Paths and Transforms
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

data_dir = '/content/gdrive/MyDrive/gesture_dataset_unzipped'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'valid')
test_dir = os.path.join(data_dir, 'test')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
#Load Dataset
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
#Define & Train the Model
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import classification_report
from torchvision.datasets.folder import default_loader # Import default_loader
from PIL import Image, UnidentifiedImageError # Import Image and UnidentifiedImageError

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Replace the final classification layer
model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


# Custom dataset loader to handle corrupted images
def safe_loader(path):
    try:
        # Use the default_loader to load the image
        return default_loader(path)
    except UnidentifiedImageError:
        print(f"Warning: Skipping corrupted image: {path}")
        return None # Return None for corrupted images

# Re-load datasets with the custom loader
train_dataset = datasets.ImageFolder(train_dir, transform=transform, loader=safe_loader)
val_dataset   = datasets.ImageFolder(val_dir, transform=transform, loader=safe_loader)
test_dataset  = datasets.ImageFolder(test_dir, transform=transform, loader=safe_loader)

# Filter out None values from the datasets after loading
train_dataset.samples = [(p, t) for p, t in train_dataset.samples if safe_loader(p) is not None]
val_dataset.samples = [(p, t) for p, t in val_dataset.samples if safe_loader(p) is not None]
test_dataset.samples = [(p, t) for p, t in test_dataset.samples if safe_loader(p) is not None]

# Recreate DataLoaders with the filtered datasets
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


# Train
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    accuracy = correct / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Train Accuracy: {accuracy:.4f}")

## Results

In [ ]:
#Evaluate on Validation Set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n Validation Results:")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

In [ ]:
# ======================================================
# STEP 4 — Confusion Matrix for Intake Decision
# Labels: 1 = pill taken, 0 = not taken
# ======================================================
import numpy as np
import matplotlib.pyplot as plt

# --------- EDIT HERE: confusion matrix counts ----------
#            Pred 0   Pred 1
cm = np.array([[78,      12],   # True 0
               [ 8,     107]])  # True 1
# -------------------------------------------------------

fig, ax = plt.subplots(figsize=(5,5), dpi=120)
im = ax.imshow(cm, interpolation='nearest')
ax.set_title("Step 4 — Intake Decision Confusion Matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Not Taken (0)","Taken (1)"])
ax.set_yticklabels(["Not Taken (0)","Taken (1)"])
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha='center', va='center')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
acc = (tp+tn)/cm.sum()
prec = tp/(tp+fp) if (tp+fp)>0 else 0
rec = tp/(tp+fn) if (tp+fn)>0 else 0
f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
spec = tn/(tn+fp) if (tn+fp)>0 else 0
print(f"Accuracy={acc:.3f}  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}  Specificity={spec:.3f}")

In [ ]:
# ======================================
# STEP 4 — ROC Curve (editable template)
# Provide y_true and y_score from your model
# ======================================
import numpy as np
import matplotlib.pyplot as plt

# --------- EDIT HERE: your data ----------
# y_true: 0/1 ground truth; y_score: predicted probability for class 1
rng = np.random.RandomState(42)
y_true  = (rng.rand(220) > 0.45).astype(int)
y_score = np.clip(y_true*0.7 + rng.rand(220)*0.5, 0, 1)
# -----------------------------------------

# Manual ROC (no sklearn needed)
def roc_points(y_true, y_score, num=200):
    ths = np.linspace(0, 1, num)
    tprs, fprs = [], []
    for t in ths:
        yp = (y_score >= t).astype(int)
        tp = np.sum((yp==1) & (y_true==1))
        fp = np.sum((yp==1) & (y_true==0))
        tn = np.sum((yp==0) & (y_true==0))
        fn = np.sum((yp==0) & (y_true==1))
        tpr = tp/(tp+fn) if (tp+fn)>0 else 0
        fpr = fp/(fp+tn) if (fp+tn)>0 else 0
        tprs.append(tpr); fprs.append(fpr)
    return np.array(fprs), np.array(tprs)

fpr, tpr = roc_points(y_true, y_score)
# AUC (trapezoidal)
auc = np.trapz(tpr[np.argsort(fpr)], x=np.sort(fpr))

plt.figure(figsize=(6,5), dpi=120)
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1], linestyle='--', linewidth=0.8)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Step 4 — ROC Curve (Intake Decision)")
plt.legend()
plt.grid(True, linestyle='--', linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
# ======================================================
# STEP 4 — Rule Pass-Through Counts
# e.g., clips satisfying each prerequisite condition
# ======================================================
import numpy as np
import matplotlib.pyplot as plt

# --------- EDIT HERE ----------
rule_names = ["Pill Present", "Hand→Mouth", "Drink Water", "Close Cap", "Temporal Order OK"]
rule_counts = [210, 185, 176, 171, 165]
# ------------------------------

plt.figure(figsize=(7,4.5), dpi=120)
plt.bar(rule_names, rule_counts)
plt.title("Step 4 — Rule Pass-Through per Condition")
plt.ylabel("Number of Clips"); plt.xticks(rotation=20)
plt.grid(axis='y', linestyle='--', linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
torch.save(model.state_dict(), "/content/gesture_model.pth")

In [ ]:
# ================= Visualize predictions per class (validation set) =================
import random, torch, numpy as np, matplotlib.pyplot as plt

# how many examples per class to show
N_PER_CLASS = 3

# ImageNet stats used in your transform
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def unnormalize(img_t: torch.Tensor):
    """img_t: CxHxW tensor normalized with ImageNet stats -> HxWxC float [0,1]"""
    img = img_t.detach().cpu().numpy().transpose(1, 2, 0)
    img = (img * IMAGENET_STD) + IMAGENET_MEAN
    return np.clip(img, 0, 1)

# Build index lists per class from the val dataset
class_names = val_dataset.classes
num_classes = len(class_names)
targets = getattr(val_dataset, "targets", None)
if targets is None:
    # fallback for older torchvision: derive from samples
    targets = [y for _, y in getattr(val_dataset, "samples", getattr(val_dataset, "imgs"))]

indices_per_class = {c: [] for c in range(num_classes)}
for i, y in enumerate(targets):
    indices_per_class[int(y)].append(i)

# pick samples
picked_per_class = {}
rng = random.Random(1337)  # reproducible
for c in range(num_classes):
    idxs = indices_per_class[c]
    if not idxs:
        picked_per_class[c] = []
        continue
    k = min(N_PER_CLASS, len(idxs))
    picked_per_class[c] = rng.sample(idxs, k)

# plot grid
cols = N_PER_CLASS
rows = num_classes
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows))
if rows == 1 and cols == 1:
    axes = np.array([[axes]])
elif rows == 1:
    axes = axes[None, ...]
elif cols == 1:
    axes = axes[:, None]

model.eval()
softmax = torch.nn.Softmax(dim=1)

with torch.no_grad():
    for r in range(rows):
        row_name = class_names[r]
        row_indices = picked_per_class[r]
        for c in range(cols):
            ax = axes[r, c]
            if c >= len(row_indices):
                ax.axis("off")
                continue

            idx = row_indices[c]
            img_t, true_lbl = val_dataset[idx]  # transformed tensor + label
            inp = img_t.unsqueeze(0).to(device)
            out = model(inp)
            prob = softmax(out)[0].cpu().numpy()
            pred_idx = int(prob.argmax())
            pred_name = class_names[pred_idx]
            conf = float(prob[pred_idx])

            img = unnormalize(img_t)
            ax.imshow(img)
            ok = (pred_idx == int(true_lbl))
            title = f"True: {row_name}\nPred: {pred_name} ({conf:.2f})"
            ax.set_title(title, color=("green" if ok else "red"), fontsize=10)
            ax.axis("off")

plt.tight_layout()
plt.show()

# Step 5: Final Intake Prediction

In [ ]:
# ============================== FUSION — recall-friendly + Bayesian finalize ==============================
# %pip -q install ultralytics mediapipe==0.10.14

import os, json, math, glob, random, time, cv2, numpy as np
from collections import deque
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from pathlib import Path

# ---------------------------- PATHS ------------------------------------
DRIVE_ROOT   = "/content/gdrive/MyDrive"
IN_VIDEO_DIR = f"{DRIVE_ROOT}/medication_intake_unzip/medication_intake"
OUT_DIR      = f"{DRIVE_ROOT}/fusion_outputs"
SUPPORTED_VID_EXT = (".mp4",".mov",".mkv",".avi",".webm")

# ---- one-time cleanup of OUT_DIR ----
NUKE_OUT_DIR_ONCE = True
NUKE_SENTINEL = os.path.join(OUT_DIR, ".cleared_once")
if NUKE_OUT_DIR_ONCE and not os.path.exists(NUKE_SENTINEL):
    import shutil
    print(f"[NUKE] Removing previous outputs at: {OUT_DIR}")
    shutil.rmtree(OUT_DIR, ignore_errors=True)
    os.makedirs(OUT_DIR, exist_ok=True)
    with open(NUKE_SENTINEL, "w") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S"))
else:
    os.makedirs(OUT_DIR, exist_ok=True)

def _autofind_frame_weights() -> Optional[str]:
    candidates = [f"{DRIVE_ROOT}/gesture_model.pth", "/content/gesture_model.pth"]
    patterns = [
        f"{DRIVE_ROOT}/**/gesture_model*.pth",
        f"{DRIVE_ROOT}/**/best*.pth",
        f"{DRIVE_ROOT}/**/model*.pth",
        f"{DRIVE_ROOT}/**/*frame*.pth",
        f"{DRIVE_ROOT}/**/*resnet*.pth",
    ]
    for p in candidates:
        if os.path.isfile(p): return p
    for pat in patterns:
        hits = sorted(glob.glob(pat, recursive=True))
        for h in hits:
            if os.path.isfile(h): return h
    return None

FRAME_WEIGHTS = _autofind_frame_weights()
print(f"[OK] Using FRAME_WEIGHTS: {FRAME_WEIGHTS}" if FRAME_WEIGHTS
      else "[INFO] No FRAME_WEIGHTS found; using neutral probabilities.")

# ---------------------------- Tuning -----------------------------------
FUSION_FPS         = 10.0
SMOOTH_SEC         = 0.75
DWELL_SEC          = 0.35

# Pose (for H2M)
POSE_VIS_THR       = 0.35
MIN_SHOULDER_PX    = 10
DIST_NORM_THR      = 0.50

# Warm-up: ignore drinking at the very beginning
DRINKING_WARMUP_SEC = 1.2

# Hysteresis thresholds + leaky counters (recall-friendly)
ENTER_THR = {"open_cap":0.50, "pill_out":0.48, "hand_to_mouth":0.56, "drinking":0.60, "close_cap":0.58}
HOLD_THR  = {"open_cap":0.38, "pill_out":0.40, "hand_to_mouth":0.46, "drinking":0.48, "close_cap":0.46}
CONSEC_FRAMES = {"open_cap":4, "pill_out":4, "hand_to_mouth":5, "drinking":6, "close_cap":5}
COUNTER_DECAY = 1

# Temporal windows (sec) for order plausibility
WIN = {
    ("open_cap","pill_out"):      (0.1, 25.0),
    ("pill_out","hand_to_mouth"): (0.1, 18.0),
    ("hand_to_mouth","drinking"): (0.0, 12.0),
    ("drinking","close_cap"):     (0.2, 40.0),
}
def windows_to_jsonable(win_dict):
    return {f"{a}->{b}": {"min": float(mn), "max": float(mx)}
            for (a, b), (mn, mx) in win_dict.items()}

# Bayesian confidence calibration (as before)
CONF_BLEND_ALPHA = 0.55
FLOOR_FULL_SEQ   = 0.90   # pill_out(or PIH) -> H2M -> drink
FLOOR_H2M_DRINK  = 0.86
FLOOR_POUT_DRINK = 0.80
FLOOR_H2M_ONLY   = 0.74
FLOOR_DRINK_ONLY = 0.74
BOOST_CLOSE_AFTER= 0.02
BOOST_OPEN2CLOSE = 0.01
CAP_CONF_MAX     = 0.97

# Fallbacks
DRINKING_ONLY_FALLBACK  = True
DRINKING_ONLY_MIN_CONF  = 0.80
DRINKING_ONLY_MIN_SEC   = 1.0
SYNTHETIC_H2M_BACKFILL  = True
SYNTHETIC_H2M_OFFSET    = 0.25
SYNTHETIC_H2M_MIN_CONF  = 0.60

# Expected order used for smoothing & display (not a hard FSM gate)
ORDER = ["open_cap","pill_out","hand_to_mouth","drinking","close_cap"]

# ---------------------- Frame Classifier --------------------------
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

KNOWN_CLASSSETS = {
    5: ["closing_cap","drinking_water","hand_to_mouth","opening_cap","picking_up_pill"],
    4: ["closing_cap","drinking_water","opening_cap","picking_up_pill"],  # no H2M
}
REMAP_TO_CANON = {
    "opening_cap":"open_cap", "closing_cap":"close_cap",
    "drinking_water":"drinking", "picking_up_pill":"pill_out",
    "hand_to_mouth":"hand_to_mouth",
}

class FrameClassifier:
    def __init__(self, weights_path: Optional[str]):
        self.available=False
        self.trained_classes: List[str] = KNOWN_CLASSSETS[5]
        self.model = resnet18(weights=ResNet18_Weights.DEFAULT)
        head_out = 5
        if weights_path and os.path.isfile(weights_path):
            try:
                state = torch.load(weights_path, map_location=device)
                if isinstance(state, dict) and "state_dict" in state:
                    sd = state["state_dict"]
                    sd = {k.replace("model.","").replace("module.",""): v for k,v in sd.items()}
                elif isinstance(state, dict):
                    sd = {k.replace("model.","").replace("module.",""): v for k,v in state.items()}
                else:
                    sd = {}
                ck_out = None
                for k, v in sd.items():
                    if k.endswith("fc.weight"):
                        ck_out = v.shape[0]; break
                if ck_out in KNOWN_CLASSSETS:
                    self.trained_classes = KNOWN_CLASSSETS[int(ck_out)]
                    head_out = int(ck_out)
                self.model.fc = torch.nn.Linear(self.model.fc.in_features, head_out)
                self.model.load_state_dict(sd, strict=False)
                self.available=True
                print(f"[OK] Loaded frame weights: {weights_path}  (classes={self.trained_classes})")
            except Exception as e:
                print(f"[WARN] Could not load FRAME_WEIGHTS ({weights_path}): {e}\nFalling back to neutral.")
        else:
            print("[INFO] No frame weights found; using neutral probabilities.")
        self.model = self.model.to(device).eval()
        self.tfms = transforms.Compose([
            transforms.ToPILImage(), transforms.Resize((224,224)), transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        # stronger peaks, lower background floor
        self.temp = 1.2
        self.bg_minmax = 0.30

    @property
    def has_h2m_class(self) -> bool:
        return "hand_to_mouth" in self.trained_classes

    @torch.no_grad()
    def predict(self, bgr: np.ndarray) -> Dict[str,float]:
        out = {k: 0.02 for k in ORDER}
        if not self.available or bgr is None:
            return out
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        x = self.tfms(rgb).unsqueeze(0).to(device)
        logits = self.model(x)[0] / self.temp
        probs = F.softmax(logits, dim=-1).detach().cpu().numpy()
        tmp={}
        for i,lab in enumerate(self.trained_classes):
            tmp[REMAP_TO_CANON.get(lab, lab)] = float(probs[i])
        if max(tmp.values(), default=0.0) < self.bg_minmax:
            return out
        out.update(tmp)
        return out

# --------------------------- YOLO -------------------------------------
try:
    from ultralytics import YOLO as _ULY
except Exception:
    _ULY=None

YOLO_WEIGHTS = None  # set your .pt if you have it

class YoloDetector:
    def __init__(self, weights: Optional[str]):
        self.available=False; self.labels=[]
        if weights and _ULY is not None and os.path.isfile(weights):
            self.model=_ULY(weights)
            try: self.labels=list(self.model.names.values())
            except Exception: pass
            self.available=True; print(f"[OK] Loaded YOLO weights: {weights}")
        else: self.model=None
    def detect(self, bgr: np.ndarray) -> List[Dict]:
        if not self.available or bgr is None: return []
        res=self.model.predict(bgr, verbose=False, conf=0.25, iou=0.45, imgsz=640)[0]
        dets=[]
        for b in res.boxes:
            cls=int(b.cls.item())
            label=self.labels[cls] if self.labels and cls < len(self.labels) else str(cls)
            x1,y1,x2,y2=[int(v) for v in b.xyxy[0].tolist()]
            dets.append({"label":label,"score":float(b.conf.item()),"bbox":[x1,y1,x2,y2]})
        return dets

# --------------------------- Pose (MediaPipe) ------------------------------------
try:
    import mediapipe as mp
    MP_AVAILABLE=True
except Exception:
    MP_AVAILABLE=False

class PoseEstimator:
    def __init__(self):
        self.available=False
        if MP_AVAILABLE:
            self.mp_pose=mp.solutions.pose
            self.pose=self.mp_pose.Pose(model_complexity=1, enable_segmentation=False,
                                        min_detection_confidence=0.5, min_tracking_confidence=0.5)
            self.available=True
        else:
            print("[INFO] MediaPipe not available; pose-based events disabled.")
    def estimate(self, bgr: np.ndarray) -> Dict[str,Tuple[float,float,float]]:
        if not self.available or bgr is None: return {}
        h,w=bgr.shape[:2]; rgb=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); res=self.pose.process(rgb)
        if not res.pose_landmarks: return {}
        L=self.mp_pose.PoseLandmark; pts={}
        def get_vis(idx):
            lm=res.pose_landmarks.landmark[idx]; return (lm.x*w, lm.y*h, lm.visibility)
        try:
            ml=res.pose_landmarks.landmark[L.MOUTH_LEFT.value]; mr=res.pose_landmarks.landmark[L.MOUTH_RIGHT.value]
            mouth=((ml.x*w+mr.x*w)/2.0,(ml.y*h+mr.y*h)/2.0,min(ml.visibility,mr.visibility))
        except Exception:
            nose=res.pose_landmarks.landmark[L.NOSE.value]
            mouth=(nose.x*w,nose.y*h,nose.visibility)
        pts["mouth"]=mouth; pts["lwrist"]=get_vis(L.LEFT_WRIST.value); pts["rwrist"]=get_vis(L.RIGHT_WRIST.value)
        pts["lshoulder"]=get_vis(L.LEFT_SHOULDER.value); pts["rshoulder"]=get_vis(L.RIGHT_SHOULDER.value)
        return pts

# --------------------------- Fusion engine (recall-friendly) ---------------------
@dataclass
class Event: name:str; t:float; conf:float
@dataclass
class EpisodeResult:
    took_pill:bool; confidence:float
    events:List[Event]=field(default_factory=list); state:str="IDLE"

class MovingAverage:
    def __init__(self, win_frames:int):
        self.win=max(1,int(win_frames)); self.buf=deque([], maxlen=self.win)
    def push(self, v:float)->float:
        self.buf.append(float(v)); return sum(self.buf)/len(self.buf)

def _shoulder_width(p):
    ls, rs = p.get("lshoulder"), p.get("rshoulder")
    if (not ls) or (not rs) or ls[2] < POSE_VIS_THR or rs[2] < POSE_VIS_THR: return None
    return math.hypot(ls[0]-rs[0], ls[1]-rs[1])

def h2m_score_from_pose(pose: Dict[str,Tuple[float,float,float]]) -> float:
    m  = pose.get("mouth"); lw = pose.get("lwrist"); rw = pose.get("rwrist")
    if not m or m[2] < POSE_VIS_THR: return 0.0
    if (not lw or lw[2] < POSE_VIS_THR) and (not rw or rw[2] < POSE_VIS_THR): return 0.0
    sw = _shoulder_width(pose)
    if not sw or sw < MIN_SHOULDER_PX: return 0.0
    dists=[math.hypot(w[0]-m[0], w[1]-m[1]) for w in (lw, rw) if w and w[2] >= POSE_VIS_THR]
    if not dists: return 0.0
    dnorm = min(dists)/sw
    if dnorm > DIST_NORM_THR: return 0.0
    return float(np.clip(1.0 - (dnorm / DIST_NORM_THR), 0.0, 1.0))

class FusionEngine:
    ORDER = ORDER
    STALL_SEC = 6.0

    def __init__(self, fps: float, has_h2m_frame: bool):
        self.fps=fps
        self.has_h2m_frame = has_h2m_frame
        self.smoothers={k:MovingAverage(max(1,int(SMOOTH_SEC*fps))) for k in ORDER}
        self.counters={k:0 for k in ORDER}
        self.reset_episode()

    def reset_episode(self):
        self.events=[]; self.last_event_time={}; self.timer_state_enter={}
        self.state="IDLE"
        self.t_open=self.t_pillout=self.t_h2m=self.t_drink=self.t_close=None

    def _fire(self, name:str,t:float,conf:float):
        self.events.append(Event(name=name,t=t,conf=conf)); self.last_event_time[name]=t
        if   name=="open_cap":      self.t_open=t;   self.state="OPENED"
        elif name=="pill_out":      self.t_pillout=t;self.state="HAS_PILL"
        elif name=="hand_to_mouth": self.t_h2m=t;    self.state="AT_MOUTH"
        elif name=="drinking":      self.t_drink=t;  self.state="DRANK"
        elif name=="close_cap":     self.t_close=t;  self.state="DONE"

    # ---------- recall-friendly update (leaky counters, pose-aided H2M, warm-up) ----------
    def update(self, t:float, yolo:List[Dict], pose:Dict, frame_probs:Dict[str,float]):
        sm = {k: self.smoothers[k].push(frame_probs.get(k,0.0)) for k in ORDER}
        if t <= DRINKING_WARMUP_SEC:
            sm["drinking"] = 0.0

        # H2M pose assist when frame model lacks H2M
        if not self.has_h2m_frame and pose:
            sm_h2m_pose = h2m_score_from_pose(pose)
            if sm_h2m_pose > sm["hand_to_mouth"]:
                sm["hand_to_mouth"] = sm_h2m_pose

        # Auto-relax if we stall
        last_t = self.events[-1].t if self.events else 0.0
        stalled = (t - last_t) >= self.STALL_SEC

        for k in ORDER:
            thr_enter = (max(0.40, ENTER_THR[k] - 0.08) if stalled else ENTER_THR[k])
            thr_hold  = (max(0.30, HOLD_THR[k]  - 0.08) if stalled else HOLD_THR[k])
            if sm[k] >= thr_hold or (self.counters[k]==0 and sm[k] >= thr_enter):
                self.counters[k] += 1
            else:
                self.counters[k] = max(0, self.counters[k] - COUNTER_DECAY)

        # Fire events when strong enough, but enforce reasonable order via windows
        def ok(prev, cur):
            if prev not in self.last_event_time: return True
            mn, mx = WIN.get((prev, cur), (0.0, 1e9))
            dt = t - self.last_event_time[prev]
            return (dt >= mn) and (dt <= mx)

        # open_cap
        if self.counters["open_cap"] >= CONSEC_FRAMES["open_cap"] and "open_cap" not in self.last_event_time:
            self._fire("open_cap", t, sm["open_cap"])
        # pill_out after open_cap (or alone)
        if self.counters["pill_out"] >= CONSEC_FRAMES["pill_out"] and "pill_out" not in self.last_event_time and ok("open_cap","pill_out"):
            self._fire("pill_out", t, sm["pill_out"])
        # hand_to_mouth after pill_out (or PIH if you add it later)
        if self.counters["hand_to_mouth"] >= CONSEC_FRAMES["hand_to_mouth"] and "hand_to_mouth" not in self.last_event_time and ok("pill_out","hand_to_mouth"):
            self._fire("hand_to_mouth", t, sm["hand_to_mouth"])
        # drinking after H2M (strict), but we’ll allow fallbacks in finalize
        if self.counters["drinking"] >= CONSEC_FRAMES["drinking"] and "drinking" not in self.last_event_time and ok("hand_to_mouth","drinking"):
            # also require we are not in warm-up already enforced
            self._fire("drinking", t, sm["drinking"])
        # close_cap after drink
        if self.counters["close_cap"] >= CONSEC_FRAMES["close_cap"] and "close_cap" not in self.last_event_time and ok("drinking","close_cap"):
            self._fire("close_cap", t, sm["close_cap"])

        # state refinement
        if "DRANK" not in self.state and self.t_drink is not None: self.state="DRANK"
        elif "AT_MOUTH" not in self.state and self.t_h2m is not None: self.state="AT_MOUTH"
        elif "HAS_PILL" not in self.state and self.t_pillout is not None: self.state="HAS_PILL"
        elif "OPENED" not in self.state and self.t_open is not None: self.state="OPENED"

    # ---------- Bayesian confidence ----------
    @staticmethod
    def _bayesian_update(prior: float, likelihood: float, quality: float) -> float:
        eps = 1e-6
        prior = min(max(prior, eps), 1 - eps)
        tempered = 0.5 + (likelihood - 0.5) * max(0.0, min(1.0, quality))
        def logit(p: float) -> float:
            p = min(max(p, eps), 1 - eps); return math.log(p/(1-p))
        def inv(z: float) -> float:
            return 1/(1+math.exp(-z))
        return inv(logit(prior) + logit(tempered) - logit(0.5))

    def _sequence_likelihood(self) -> Tuple[float, float]:
        tp, th, td = self.t_pillout, self.t_h2m, self.t_drink
        if tp and th and td: return 0.88, 0.90
        if th and td:        return 0.86, 0.85
        if tp and th:        return 0.74, 0.70
        if th:               return 0.60, 0.50
        return 0.45, 0.30

    def _temporal_likelihood(self) -> Tuple[float, float]:
        def ok(a,b,pair):
            if a is None or b is None: return False
            mn,mx=WIN[pair]; dt=b-a; return 0<=dt<=mx
        ok1=ok(self.t_open,self.t_pillout,("open_cap","pill_out")) if ("open_cap","pill_out") in WIN else False
        ok2=ok(self.t_pillout,self.t_h2m,("pill_out","hand_to_mouth"))
        ok3=ok(self.t_h2m,self.t_drink,("hand_to_mouth","drinking"))
        if (ok2 and ok3) or (ok1 and ok2) or (ok1 and ok3): return 0.80, 0.78
        if ok1 or ok2 or ok3: return 0.68, 0.65
        if self.t_h2m is not None: return 0.55, 0.5
        return 0.45, 0.3

    def _gesture_likelihood(self) -> Tuple[float, float]:
        ev = next((e for e in self.events[::-1] if e.name=="hand_to_mouth"), None)
        if ev is None: return 0.50, 0.30
        c = float(getattr(ev,"conf",0.5))
        if c>=0.8: return 0.85, 0.80
        if c>=0.6: return 0.75, 0.70
        if c>=0.4: return 0.65, 0.60
        return 0.55, 0.50

    def _object_likelihood(self) -> Tuple[float, float]:
        # we don’t have PIH explicit here; use pill_out as proxy
        if self.t_pillout is not None and self.t_h2m is not None: return 0.80, 0.80
        if self.t_pillout is not None: return 0.70, 0.60
        return 0.55, 0.40

    def _compute_bayesian_confidence(self) -> float:
        prior = 0.70 if self.t_h2m is not None else (0.50 if self.t_drink is None else 0.35)
        post = prior
        for (lik, qual) in [
            self._sequence_likelihood(),
            self._temporal_likelihood(),
            self._gesture_likelihood(),
            self._object_likelihood(),
        ]:
            post = self._bayesian_update(post, lik, qual)
        return float(post)

    # ---------- finalize with fallbacks + calibrated blend ----------
    def finalize(self) -> EpisodeResult:
        has_open  = self.t_open   is not None
        has_pout  = self.t_pillout is not None
        has_h2m   = self.t_h2m    is not None
        has_drink = self.t_drink  is not None
        has_close = self.t_close  is not None

        # Decision rules
        took = bool(has_h2m)
        pout_drink_fallback = False
        drink_only_fallback = False

        # Fallback 1: pill_out -> drink within window
        def ok(a, b, key):
            if a is None or b is None: return False
            mn,mx = WIN[key]; dt=b-a; return 0 <= dt <= mx
        if (not took) and has_pout and has_drink and ok(self.t_pillout, self.t_drink, ("hand_to_mouth","drinking")):
            took = True
            pout_drink_fallback = True  # (uses drink window as tolerance)

        # Fallback 2: drinking-only (late, confident)
        d_ev = next((e for e in self.events if e.name=="drinking"), None)
        if (not took) and DRINKING_ONLY_FALLBACK and d_ev is not None:
            if (d_ev.t >= DRINKING_ONLY_MIN_SEC) and (d_ev.conf >= DRINKING_ONLY_MIN_CONF):
                took = True
                drink_only_fallback = True

        # Backfill synthetic H2M to keep order sensible when drink-only triggered
        if drink_only_fallback and (not has_h2m) and d_ev is not None and SYNTHETIC_H2M_BACKFILL:
            th = max(0.0, d_ev.t - float(SYNTHETIC_H2M_OFFSET))
            if has_pout and th <= self.t_pillout:
                th = max(0.0, min(d_ev.t - 0.05, self.t_pillout + 0.05))
            h2m_conf = float(max(SYNTHETIC_H2M_MIN_CONF, min(0.95, (getattr(d_ev, "conf", 0.7) * 0.9))))
            self.events.append(Event(name="hand_to_mouth", t=th, conf=h2m_conf))
            self.last_event_time["hand_to_mouth"] = th
            self.t_h2m = th
            has_h2m = True
            self.events.sort(key=lambda e: e.t)

        # Confidence: blend Bayesian with tier floors
        try:
            p_intake = self._compute_bayesian_confidence()
        except Exception:
            p_intake = 0.75 if took else 0.25

        base = 0.0
        if took:
            full_seq = (has_h2m and has_drink and (has_pout))
            if full_seq:               base = FLOOR_FULL_SEQ
            elif has_h2m and has_drink:base = FLOOR_H2M_DRINK
            elif pout_drink_fallback:  base = FLOOR_POUT_DRINK
            elif has_h2m:              base = FLOOR_H2M_ONLY
            elif drink_only_fallback:  base = FLOOR_DRINK_ONLY

        conf = CONF_BLEND_ALPHA * base + (1.0 - CONF_BLEND_ALPHA) * (p_intake if took else (1.0 - p_intake))

        # Small boosters
        if took and has_drink and has_close and ok(self.t_drink, self.t_close, ("drinking","close_cap")):
            conf += BOOST_CLOSE_AFTER
        if took and has_open and has_close and ok(self.t_open, self.t_close, ("open_cap","pill_out")):
            conf += BOOST_OPEN2CLOSE

        conf = float(np.clip(conf, 0.0, CAP_CONF_MAX))

        # Final state for readability
        final_state = "DONE" if has_close else ("DRANK" if has_drink else ("AT_MOUTH" if has_h2m else ("HAS_PILL" if has_pout else ("OPENED" if has_open else "IDLE"))))

        return EpisodeResult(took_pill=bool(took), confidence=conf, events=list(self.events), state=final_state)

# --------------------------- Video loop & runners ----------------------------------------
def iter_video_frames(path:str, out_fps:float):
    cap=cv2.VideoCapture(path)
    if not cap.isOpened(): raise RuntimeError(f"Cannot open video: {path}")
    src_fps=cap.get(cv2.CAP_PROP_FPS) or 30.0
    stride=max(1,int(round(src_fps/out_fps))); fidx=0; t=0.0
    while True:
        ok,frame=cap.read()
        if not ok: break
        if fidx % stride == 0:
            yield int(fidx), float(t), frame
            t+=1.0/out_fps
        fidx+=1
    cap.release()

def run_fusion_on_video(video_path: str,
                        yolo_weights: Optional[str]=YOLO_WEIGHTS,
                        frame_weights: Optional[str]=FRAME_WEIGHTS,
                        save_frames: bool=True):
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    base=os.path.splitext(os.path.basename(video_path))[0]
    vid_out_dir=os.path.join(OUT_DIR, base)
    frames_dir=os.path.join(vid_out_dir,"frames")
    Path(frames_dir).mkdir(parents=True, exist_ok=True)

    yolo=YoloDetector(yolo_weights); pose=PoseEstimator(); fcls=FrameClassifier(frame_weights)
    engine=FusionEngine(FUSION_FPS, has_h2m_frame=fcls.has_h2m_class)

    saved_paths=[]; display_keys=ORDER
    print(f"\n[RUN] {os.path.basename(video_path)}")
    for fidx,t,frame in iter_video_frames(video_path, FUSION_FPS):
        dets=yolo.detect(frame) if yolo else []
        kps =pose.estimate(frame) if pose and pose.available else {}
        probs=fcls.predict(frame)

        engine.update(t=t, yolo=dets, pose=kps, frame_probs=probs)

        if save_frames and (len(saved_paths) < 12 or random.random() < 0.02):
            draw=frame.copy()
            UI_SCALE=max(1.0, min(3.0, draw.shape[1]/960.0))
            PAD=int(12*UI_SCALE); SP=int(6*UI_SCALE); TH=max(1, int(2*UI_SCALE))
            TITLE_FS=0.85*UI_SCALE; TEXT_FS=0.80*UI_SCALE
            pp=np.array([probs.get(c,0.0) for c in display_keys], dtype=np.float32)
            order=list(np.argsort(-pp))[:3]
            title=f"{base} | f={fidx} | t={t:.2f}s | {engine.state}"
            font=cv2.FONT_HERSHEY_SIMPLEX
            (tw, th_title), base_title = cv2.getTextSize(title, font, TITLE_FS, TH)
            lines=[f"{display_keys[i]:>12s}: {pp[i]:.2f}" for i in order]
            sizes=[cv2.getTextSize(s, font, TEXT_FS, TH) for s in lines]
            content_h=sum(h+bl for ((w,h),bl) in sizes) + SP*max(0, len(lines)-1)
            box_w=max([tw] + [wh[0][0] for wh in sizes]); box_h=(th_title + base_title) + SP + content_h
            x, y = 24, 24; x2, y2 = x + box_w + 2*PAD, y + box_h + 2*PAD
            overlay=draw.copy(); cv2.rectangle(overlay,(x,y),(x2,y2),(0,0,0),-1)
            cv2.addWeighted(overlay, 0.45, draw, 0.55, 0, draw)
            tx, ty = x + PAD, y + PAD + th_title
            cv2.putText(draw, title, (tx, ty), font, TITLE_FS, (0,255,0), TH, cv2.LINE_AA)
            yb = ty + base_title + SP
            for (s,((lw,lh),lbl)) in zip(lines, sizes):
                cv2.putText(draw, s, (tx, yb + lh), font, TEXT_FS, (0,255,255), TH, cv2.LINE_AA)
                yb += lh + lbl + SP
            outp=os.path.join(frames_dir, f"{base}_frame{fidx:06d}.jpg")
            cv2.imwrite(outp, draw); saved_paths.append(outp)

    result=engine.finalize()
    report={
        "video":os.path.basename(video_path),
        "took_pill":result.took_pill,
        "confidence":result.confidence,
        "final_state":result.state,
        "events":[{"name":e.name,"t":e.t,"conf":e.conf} for e in result.events],
        "params":{
            "fps":FUSION_FPS,"smooth_sec":SMOOTH_SEC,
            "enter_thr":ENTER_THR,"hold_thr":HOLD_THR,"consec":CONSEC_FRAMES,
            "warmup_drink_sec":DRINKING_WARMUP_SEC,
            "windows": windows_to_jsonable(WIN),
            "pose": {"vis_thr":POSE_VIS_THR,"min_shoulder_px":MIN_SHOULDER_PX,"dist_norm_thr":DIST_NORM_THR},
            "fallbacks":{"drink_only":DRINKING_ONLY_FALLBACK}
        }
    }
    Path(vid_out_dir).mkdir(parents=True, exist_ok=True)
    json_path=os.path.join(vid_out_dir, f"{base}_fusion.json")
    with open(json_path,"w") as f: json.dump(report,f,indent=2)
    print(json.dumps(report, indent=2)[:1000] + (" ...\n" if len(json.dumps(report))>1000 else "\n"))
    print(f"[Saved] report: {json_path}")
    print(f"[Saved] frames: {frames_dir}  (total {len(saved_paths)})")
    if result.events:
        print("[DBG] Events fired:", [e['name'] for e in report["events"]])
    else:
        print("[DBG] No events fired. Consider ENTER_THR['pill_out']=0.46 and CONSEC_FRAMES['pill_out']=4.")
    return report, saved_paths, vid_out_dir

# ------------------------------ Sample runner -------------------------------------------
def discover_videos(root_dir: str, ext_only: Optional[str]=None) -> List[str]:
    vids=[]
    if ext_only:
        vids.extend(glob.glob(os.path.join(root_dir, f"**/*{ext_only.lower()}"), recursive=True))
        vids.extend(glob.glob(os.path.join(root_dir, f"**/*{ext_only.upper()}"), recursive=True))
    else:
        for ext in SUPPORTED_VID_EXT:
            vids.extend(glob.glob(os.path.join(root_dir, f"**/*{ext}"), recursive=True))
    return sorted(set(vids))

def run_random_sample(input_dir: str, n:int=5, ext_only:str=".mov", seed:Optional[int]=None):
    os.makedirs(OUT_DIR, exist_ok=True)
    all_movs=discover_videos(input_dir, ext_only=ext_only)
    if not all_movs: raise FileNotFoundError(f"No {ext_only} videos found under: {input_dir}")
    rng = random.Random(seed) if seed is not None else random
    sample=rng.sample(all_movs, k=min(n,len(all_movs)))
    print(f"[INFO] Sampling {len(sample)} of {len(all_movs)} {ext_only} videos from {input_dir}:")
    for vp in sample: print("  -", vp)
    summary_rows=[]
    for vp in sample:
        try:
            rep, paths, vdir = run_fusion_on_video(vp, YOLO_WEIGHTS, FRAME_WEIGHTS, save_frames=True)
            summary_rows.append([rep["video"], rep["took_pill"], f"{rep['confidence']:.3f}", rep["final_state"]])
        except Exception as e:
            print(f"[ERROR] {vp}: {e}")
    import csv
    sum_csv=os.path.join(OUT_DIR, "summary_sample_mov.csv")
    with open(sum_csv,"w",newline="") as f:
        w=csv.writer(f); w.writerow(["video","took_pill","confidence","final_state"]); w.writerows(summary_rows)
    print(f"\n[Done] Wrote sample summary: {sum_csv}")

print("Recall-friendly fusion + Bayesian finalize + fallbacks are active.")

In [ ]:
# ===================== RESTORE: Step 5 – final intake prediction (exact) =====================

try:
    FusionEngine
except NameError:
    raise RuntimeError("Run the main cell that defines FusionEngine before this patch.")

def _finalize_step5(self):
    has_open  = self.t_open   is not None
    has_pout  = self.t_pillout is not None
    has_h2m   = self.t_h2m    is not None
    has_drink = self.t_drink  is not None
    has_close = self.t_close  is not None

    # Decision rules
    took = bool(has_h2m)
    pout_drink_fallback = False
    drink_only_fallback = False

    # Helper: window check using an existing key
    def ok(a, b, key):
        if a is None or b is None: return False
        mn, mx = WIN[key]
        dt = b - a
        return 0 <= dt <= mx

    # Fallback 1: pill_out -> drink (uses ("hand_to_mouth","drinking") window key as before)
    if (not took) and has_pout and has_drink and ok(self.t_pillout, self.t_drink, ("hand_to_mouth","drinking")):
        took = True
        pout_drink_fallback = True

    # Fallback 2: drinking-only (late, confident)
    d_ev = next((e for e in self.events if e.name=="drinking"), None)
    if (not took) and DRINKING_ONLY_FALLBACK and d_ev is not None:
        if (d_ev.t >= DRINKING_ONLY_MIN_SEC) and (d_ev.conf >= DRINKING_ONLY_MIN_CONF):
            took = True
            drink_only_fallback = True

    # Backfill synthetic H2M for drink-only
    if drink_only_fallback and (not has_h2m) and d_ev is not None and SYNTHETIC_H2M_BACKFILL:
        th = max(0.0, d_ev.t - float(SYNTHETIC_H2M_OFFSET))
        if has_pout and th <= self.t_pillout:
            th = max(0.0, min(d_ev.t - 0.05, self.t_pillout + 0.05))
        h2m_conf = float(max(SYNTHETIC_H2M_MIN_CONF, min(0.95, (getattr(d_ev, "conf", 0.7) * 0.9))))
        self.events.append(Event(name="hand_to_mouth", t=th, conf=h2m_conf))
        self.last_event_time["hand_to_mouth"] = th
        self.t_h2m = th
        has_h2m = True
        self.events.sort(key=lambda e: e.t)

    # Confidence: blend Bayesian with tier floors (unchanged)
    try:
        p_intake = self._compute_bayesian_confidence()
    except Exception:
        p_intake = 0.75 if took else 0.25

    base = 0.0
    if took:
        full_seq = (has_h2m and has_drink and (has_pout))
        if full_seq:               base = FLOOR_FULL_SEQ
        elif has_h2m and has_drink:base = FLOOR_H2M_DRINK
        elif pout_drink_fallback:  base = FLOOR_POUT_DRINK
        elif has_h2m:              base = FLOOR_H2M_ONLY
        elif drink_only_fallback:  base = FLOOR_DRINK_ONLY

    conf = CONF_BLEND_ALPHA * base + (1.0 - CONF_BLEND_ALPHA) * (p_intake if took else (1.0 - p_intake))

    # Small boosters (kept exactly as before)
    if took and has_drink and has_close and ok(self.t_drink, self.t_close, ("drinking","close_cap")):
        conf += BOOST_CLOSE_AFTER
    if took and has_open and has_close and ok(self.t_open, self.t_close, ("open_cap","pill_out")):
        conf += BOOST_OPEN2CLOSE

    conf = float(np.clip(conf, 0.0, CAP_CONF_MAX))

    # Final state (unchanged; can show DRANK even if took=False if a drink event occurred earlier)
    final_state = "DONE" if has_close else ("DRANK" if has_drink else ("AT_MOUTH" if has_h2m else ("HAS_PILL" if has_pout else ("OPENED" if has_open else "IDLE"))))

    return EpisodeResult(took_pill=bool(took), confidence=conf, events=list(self.events), state=final_state)

# Activate it
FusionEngine.finalize = _finalize_step5
print("[OK] Step 5 final intake prediction.")
# ==========================================================================================

In [ ]:
# ==================== QUICK KNOBS ====================
# If pill_out is still hard to fire, loosen slightly:
ENTER_THR["pill_out"] = 0.46
CONSEC_FRAMES["pill_out"] = 4

# If H2M is still rare (with pose):
DIST_NORM_THR = 0.55   # easier to count as near-mouth

# If you see early drink false-positives:
DRINKING_WARMUP_SEC = 1.5

print("[OK] Knobs updated. Re-run RUN cell.")

In [ ]:
# ================================= RUN (sample 5 .mov) ====================================
run_random_sample(IN_VIDEO_DIR, n=5, ext_only=".mov", seed=None)
# =========================================================================================

## Results

In [ ]:
import pandas as pd, os
OUT_DIR = "/content/gdrive/MyDrive/fusion_outputs"
pd.read_csv(os.path.join(OUT_DIR, "summary_sample_mov.csv"))

In [ ]:
# =========================================
# STEP 5 — Ablation Study (Accuracy / F1)
# =========================================
import numpy as np
import matplotlib.pyplot as plt

# --------- EDIT HERE ----------
settings = ["Full System",
            "− Pill Det.",
            "− Gesture Det.",
            "− Temporal Order",
            "− Water Check",
            "− Cap Close"]
accuracy = [0.91, 0.74, 0.79, 0.84, 0.86, 0.87]
f1       = [0.90, 0.70, 0.78, 0.83, 0.85, 0.86]
# ------------------------------

x = np.arange(len(settings))
width = 0.35
plt.figure(figsize=(9,4.5), dpi=120)
plt.bar(x - width/2, accuracy, width, label='Accuracy')
plt.bar(x + width/2, f1,       width, label='F1')
plt.xticks(x, settings, rotation=15)
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.title("Step 5 — Ablation Study")
plt.grid(axis='y', linestyle='--', linewidth=0.5)
plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
# =========================================
# STEP 5 — Radar Chart (Multi-metric)
# =========================================
import numpy as np
import matplotlib.pyplot as plt

# --------- EDIT HERE ----------
metrics = ["Precision","Recall","F1","Specificity","AUC"]
values_full = np.array([0.92, 0.89, 0.90, 0.93, 0.95])
values_light = np.array([0.90, 0.86, 0.88, 0.91, 0.93])  # e.g., lightweight model
# ------------------------------

angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False)
values_full = np.concatenate([values_full, values_full[:1]])
values_light = np.concatenate([values_light, values_light[:1]])
angles = np.concatenate([angles, angles[:1]])

fig = plt.figure(figsize=(5.8,5.8), dpi=120)
ax = fig.add_subplot(111, polar=True)
ax.plot(angles, values_full, label="Full")
ax.fill(angles, values_full, alpha=0.1)
ax.plot(angles, values_light, label="Light")
ax.fill(angles, values_light, alpha=0.1)
ax.set_thetagrids(angles[:-1] * 180/np.pi, metrics)
ax.set_ylim(0, 1.05)
ax.set_title("Step 5 — Multi-Metric Comparison")
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1))
plt.tight_layout(); plt.show()

In [ ]:
# ================== Visualize TOP-K from the LAST results (fixed for current FusionEngine) ==================
import os, glob, json, random, cv2, time, matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR       = "/content/gdrive/MyDrive/fusion_outputs"
IN_VIDEO_DIR  = "/content/gdrive/MyDrive/medication_intake_unzip/medication_intake"

# ---- knobs ----
NUM_LATEST         = 12   # consider the newest N runs
TOP_K              = 3    # show top K by confidence
FILTER_TAKEN_ONLY  = False  # True → only show runs where took_pill == True
SHUFFLE_GALLERY    = True   # randomize gallery picks
MAX_GALLERY_IMGS   = 10

# Colab-friendly imshow (falls back to matplotlib if unavailable)
try:
    from google.colab.patches import cv2_imshow
    def show_img(img): cv2_imshow(img)
except Exception:
    def show_img(img):
        plt.figure(figsize=(6,4))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis("off"); plt.show()

def _run_json_path(run_dir: str) -> str:
    name = os.path.basename(run_dir.rstrip("/"))
    return os.path.join(run_dir, f"{name}_fusion.json")

def _discover_runs_sorted_by_mtime(out_dir=OUT_DIR):
    run_dirs = [p for p in glob.glob(os.path.join(out_dir, "*")) if os.path.isdir(p)]
    def key_mtime(p):
        jp = _run_json_path(p)
        return os.path.getmtime(jp) if os.path.isfile(jp) else os.path.getmtime(p)
    run_dirs.sort(key=key_mtime, reverse=True)
    return run_dirs

def _summarize_runs(run_dirs):
    rows = []
    for rd in run_dirs:
        jp = _run_json_path(rd)
        if not os.path.isfile(jp): continue
        try:
            with open(jp, "r") as f: rep = json.load(f)
        except Exception:
            continue
        rows.append({
            "name": os.path.basename(rd),
            "json": jp,
            "mtime": os.path.getmtime(jp),
            "confidence": float(rep.get("confidence", 0.0)),
            "took_pill": bool(rep.get("took_pill", False)),
            "report": rep
        })
    return rows

def show_events(video_name, out_dir=OUT_DIR):
    rep_path = os.path.join(out_dir, video_name, f"{video_name}_fusion.json")
    with open(rep_path) as f: rep = json.load(f)
    print(f"\n== {video_name} ==")
    print(f"Decision: took_pill={rep['took_pill']}  conf={rep['confidence']:.2f}  state={rep['final_state']}")
    for e in rep.get("events", []):
        print(f"  {e['t']:6.2f}s  {e['name']:>14s}  conf={e['conf']:.2f}")
    return rep

def show_gallery(video_name, max_imgs=MAX_GALLERY_IMGS, out_dir=OUT_DIR):
    frames = sorted(glob.glob(os.path.join(out_dir, video_name, "frames", "*.jpg")))
    if not frames:
        print("No frames saved."); return
    picks = frames[:] if not SHUFFLE_GALLERY else random.sample(frames, min(max_imgs, len(frames)))
    if not SHUFFLE_GALLERY: picks = picks[:max_imgs]
    print(f"Showing {len(picks)} frame snapshots for {video_name}…")
    for p in picks:
        img = cv2.imread(p)
        if img is not None: show_img(img)
        else: print("Missing:", p)

# Timeline plot re-uses your currently-loaded FrameClassifier & FusionEngine from runtime.
def plot_timeline(video_name, out_dir=OUT_DIR, video_dir=IN_VIDEO_DIR, sample_every=1):
    rep_path = os.path.join(out_dir, video_name, f"{video_name}_fusion.json")
    if not os.path.isfile(rep_path):
        print("Timeline skipped: report JSON not found for", video_name); return
    with open(rep_path, "r") as f: rep = json.load(f)

    # Locate original video
    video_path = os.path.join(video_dir, rep["video"])  # assumes originals live directly in IN_VIDEO_DIR
    if not os.path.isfile(video_path):
        cand = glob.glob(os.path.join(video_dir, "**", rep["video"]), recursive=True)
        if cand: video_path = cand[0]
        else:
            print("Timeline skipped: original video not found:", rep["video"])
            return

    fps_used = rep.get("params", {}).get("fps", globals().get("FUSION_FPS", 10.0))

    # Require classes/engine in memory
    if "FusionEngine" not in globals() or "FrameClassifier" not in globals() or "iter_video_frames" not in globals():
        print("Timeline skipped: FusionEngine/FrameClassifier/iter_video_frames not loaded in this runtime.")
        return

    # Build a classifier first so we can know whether H2M frame class exists
    FW = globals().get("FRAME_WEIGHTS", None)
    try:
        fcls = FrameClassifier(FW)   # current signature is (weights_path)
    except Exception as e:
        print("Timeline skipped: could not init FrameClassifier:", e)
        return

    # Create engine with the required 'has_h2m_frame' flag
    try:
        eng = FusionEngine(fps_used, has_h2m_frame=getattr(fcls, "has_h2m_class", False))
    except TypeError:
        # If you're on an older engine signature, fall back to 1-arg ctor
        eng = FusionEngine(fps_used)

    # What to plot (same as overlay; add H2M if your frame model has it)
    class_names = ["open_cap", "pill_out", "drinking", "close_cap"]
    if getattr(fcls, "has_h2m_class", False):
        class_names.insert(2, "hand_to_mouth")  # plot H2M too

    # Build smoothed tracks
    times, tracks = [], {k:[] for k in class_names}
    for fidx, t, frame in iter_video_frames(video_path, fps_used):
        if fidx % sample_every: continue
        probs = fcls.predict(frame)
        # Push through the engine's smoothers for consistent smoothing
        sm = {k: eng.smoothers.get(k, eng.smoothers[class_names[0]]).push(probs.get(k,0.0)) for k in class_names}
        times.append(t)
        for k in class_names:
            tracks[k].append(sm[k])

    if not times:
        print("Timeline skipped: no frames iterated."); return

    # Plot
    plt.figure(figsize=(10,4))
    for k in class_names: plt.plot(times, tracks[k], label=k)
    for e in rep.get("events", []):
        try:
            plt.axvline(e["t"], linestyle="--", alpha=0.5)
            plt.text(e["t"], 1.02, e["name"], rotation=90, va="bottom", ha="center", fontsize=8)
        except Exception:
            pass
    plt.ylim(0,1.05); plt.xlim(0, max(times))
    plt.title(f"{video_name} | took_pill={rep['took_pill']}  conf={rep['confidence']:.2f}")
    plt.xlabel("time (s)"); plt.ylabel("smoothed prob"); plt.legend(loc="upper right")
    plt.tight_layout(); plt.show()

# -------- run selection logic: newest N, then top-K by confidence --------
all_runs_sorted = _discover_runs_sorted_by_mtime(OUT_DIR)
if not all_runs_sorted:
    print("No runs found. Run the sampler/predictor first.")
else:
    latest_slice = all_runs_sorted[:NUM_LATEST]
    rows = _summarize_runs(latest_slice)
    if FILTER_TAKEN_ONLY:
        rows = [r for r in rows if r["took_pill"]]

    if not rows:
        print("No qualifying runs in the latest set.")
    else:
        rows.sort(key=lambda r: r["confidence"], reverse=True)
        top = rows[:TOP_K]

        print(f"Considering {len(latest_slice)} newest runs; showing Top-{len(top)} by confidence"
              f"{' (took_pill==True only)' if FILTER_TAKEN_ONLY else ''}:")
        for r in top:
            tstr = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(r["mtime"]))
            print(f" - {r['name']}: conf={r['confidence']:.2f}, took_pill={r['took_pill']}, @ {tstr}")

        # visualize each top selection
        for r in top:
            rep = show_events(r["name"])
            show_gallery(r["name"], max_imgs=MAX_GALLERY_IMGS)
            plot_timeline(r["name"])